# 02 · Why `F(W)` behaves the way it does — a dynamical-systems view

The forward pass **is a dynamical system**: a vector is repeatedly hit by a random
matrix and rectified,

$$ z_{\ell+1} = \mathrm{ReLU}(W_\ell z_\ell),\qquad \ell = 0,\dots,K-1, $$

i.e. a **product of random matrices** acting on a distribution. Everything the
contest throws at us — heavy tails, concentration with width, the width-helps /
depth-hurts asymmetry, and why `F` is *learnable at all* — falls out of this
picture. This notebook makes each piece interactive. *(Tiny/CPU by default; turn
`SCALE` up to see the effects sharpen.)*

In [ ]:
import sys
import numpy as np
import torch
import plotly.graph_objects as go
from ipywidgets import interact, IntSlider, FloatSlider

sys.path.insert(0, ".")
from config import PRESETS, describe

SCALE = "tiny"
cfg = PRESETS[SCALE]
dev = "cpu"
torch.manual_seed(0)
print(f"SCALE={SCALE!r} ({dev});  {describe(cfg)}")

def sample_weights(seed, d, K, var=None):                        # He init, W ~ N(0, 2/d)
    var = 2.0 / d if var is None else var
    g = torch.Generator().manual_seed(seed)
    return (torch.randn(K, d, d, generator=g, dtype=torch.float64) * var ** 0.5).float()

def layer_trace(W, X):
    # push X:(n,d) through W:(K,d,d), returning the activation at EVERY layer.
    # returns list of (n,d) tensors, length K+1 (input + each layer's output).
    z = X; trace = [z]
    for i in range(W.shape[0]):
        z = torch.relu(z @ W[i].T)                               # z <- ReLU(W_i z)
        trace.append(z)
    return trace

d, K = cfg["width"], cfg["depth"]

## 1. The gain trajectory — a finite-time Lyapunov exponent

Track how the activation magnitude $\|z_\ell\|$ evolves with depth. For a product
of random matrices the **log-magnitude grows roughly linearly** in depth,

$$ \tfrac1\ell \log \frac{\|z_\ell\|}{\|z_0\|} \;\to\; \lambda \quad(\text{finite-time Lyapunov exponent}), $$

and $\lambda$ **varies network to network** — some realizations expand (chaotic),
some contract (ordered). He initialization sits near the *edge of chaos* ($\lambda
\approx 0$), so magnitudes are roughly preserved on average. Each thin line below
is one random network's mean log-gain trajectory; the spread across lines is the
spread of $\lambda$ — the seed of everything that follows.

In [ ]:
def gain_trajectories(width=d, depth=K, n_nets=40, var_scale=1.0):
    fig = go.Figure()
    logg = []
    for s in range(n_nets):
        W = sample_weights(s, width, depth, var=var_scale * 2.0 / width)
        g = torch.Generator().manual_seed(100 + s)
        X = torch.randn(cfg["mc_samples"], width, generator=g)
        tr = layer_trace(W, X)
        norm = torch.stack([t.norm(dim=1).mean() for t in tr])   # mean ||z_l|| per layer
        lg = torch.log(norm / norm[0] + 1e-12).numpy()
        logg.append(lg)
        fig.add_trace(go.Scatter(y=lg, mode="lines", line=dict(width=1), opacity=0.4, showlegend=False))
    mean_lg = np.mean(logg, 0)
    fig.add_trace(go.Scatter(y=mean_lg, mode="lines+markers", line=dict(color="crimson", width=3),
                             name="mean over networks"))
    fig.update_layout(title=f"Mean log-gain vs depth (d={width}, var={var_scale}×He): slope ≈ Lyapunov λ",
                      xaxis_title="layer ℓ", yaxis_title="log ||z_ℓ|| / ||z_0||",
                      template="plotly_white", height=420)
    fig.show()
    print(f"final-layer log-gain: mean {mean_lg[-1]:.2f}, spread(std) {np.std([g[-1] for g in logg]):.2f}")

interact(gain_trajectories, width=IntSlider(d, 2, 64), depth=IntSlider(max(K, 12), 2, 40),
         var_scale=FloatSlider(value=1.0, min=0.5, max=2.0, step=0.1));

**At `var = 1×`He the mean slope is ≈ 0** (edge of chaos — magnitudes preserved).
Nudge `var_scale` above 1 and every trajectory bends upward (chaotic/expanding);
below 1 they collapse (ordered/contracting). That single slider is the ordered ↔
chaotic phase transition.

## 2. `F` is heavy-tailed — because the gain is a *product*

`‖F(W)‖` is set by the accumulated gain, and a product of `K` random factors is
**log-normal-ish with a heavy tail**: a few networks with large $\lambda$ dominate.
That heavy tail is exactly what made the raw target hard to regress (we learned in
$\log(1+F)$ space for this reason). As **width grows, the tail collapses** — the
per-layer gain self-averages over more neurons (concentration). Watch `log₁₀‖F‖`:

In [ ]:
def tail_plot(depth=K, n_nets=250):
    fig = go.Figure()
    for w in [4, 8, 16, 32]:
        norms = []
        for s in range(n_nets):
            W = sample_weights(s, w, depth)
            g = torch.Generator().manual_seed(7 + s)
            X = torch.randn(cfg["mc_samples"], w, generator=g)
            F = layer_trace(W, X)[-1].mean(0)                    # (w,)
            norms.append(float(F.norm()))
        fig.add_trace(go.Histogram(x=np.log10(np.array(norms) + 1e-9), name=f"d={w}",
                                   opacity=0.55, nbinsx=40))
    fig.update_layout(barmode="overlay", title=f"Heavy tail of log10 ‖F‖ collapses as width grows (K={depth})",
                      xaxis_title="log10 ‖F(W)‖", yaxis_title="count", template="plotly_white", height=400)
    fig.show()

interact(tail_plot, depth=IntSlider(max(K, 8), 2, 32));

## 3. Dead networks — why *width* keeps deep ReLU nets alive

A ReLU layer keeps an input alive only if some coordinate stays positive. Track the
**fraction of inputs still nonzero** at each layer. At small width the surviving
fraction **halves every couple of layers** and a deep-enough narrow network is
*100% dead* (`F ≡ 0`). Width provides redundancy — the chance *all* coordinates die
shrinks fast. (Key subtlety: this is **scale-invariant** — ReLU is positively
homogeneous, so rescaling the weights never changes *which* coordinates are
positive. No initialization fixes a too-narrow network; only width does. This is
why our toy uses width ≥ 8, and why the competition's width 256 is safely alive.)

In [ ]:
def survival_plot(depth=24):
    fig = go.Figure()
    for w in [2, 4, 8, 16, 32]:
        alive = np.zeros(depth + 1)
        S = 20
        for s in range(S):
            W = sample_weights(s, w, depth)
            g = torch.Generator().manual_seed(3 + s)
            X = torch.randn(cfg["mc_samples"], w, generator=g)
            for l, z in enumerate(layer_trace(W, X)):
                alive[l] += float((z.abs().sum(1) > 0).float().mean())
        fig.add_trace(go.Scatter(y=alive / S, mode="lines+markers", name=f"d={w}"))
    fig.update_layout(title="Fraction of inputs still alive vs depth (width is the cure for dying ReLU)",
                      xaxis_title="layer ℓ", yaxis_title="alive fraction",
                      template="plotly_white", height=420)
    fig.show()

interact(survival_plot, depth=IntSlider(24, 4, 40));

## 4. Why `F` is *learnable*: `f` is kinked, but `F` is smooth

`f(x;W)` is only **continuous** in `W` (each ReLU adds a kink where a unit switches
on/off). But `F(W)=\mathbb{E}_x[f]` **averages over `x`**, and different inputs kink
at different places — so the kinks wash out and **`F` is smooth** (differentiable).
That smoothness is exactly what lets a network learn `F`.

Perturb the weights along a direction, `W + tΔW`, and plot one output coordinate.
Thin lines are `f` for a few *fixed* inputs — piecewise-linear, kinked. The bold
line is `F = E_x[f]` — smooth. This is the whole reason `01_the_method` can work.

In [ ]:
def smoothness_scan(seed=0, coord=0, t_max=3.0, n_fixed=5):
    W0 = sample_weights(seed, d, K)
    g = torch.Generator().manual_seed(seed + 1)
    dW = torch.randn(K, d, d, generator=g); dW = dW / dW.norm()   # unit direction
    ts = torch.linspace(-t_max, t_max, 300)
    x_few = torch.randn(n_fixed, d, generator=torch.Generator().manual_seed(seed + 2))
    x_mc = torch.randn(cfg["mc_samples"], d, generator=torch.Generator().manual_seed(seed + 3))
    f_curves = torch.empty(len(ts), n_fixed); F_curve = torch.empty(len(ts))
    for j, t in enumerate(ts):
        Wt = W0 + t * dW
        f_curves[j] = layer_trace(Wt, x_few)[-1][:, coord]
        F_curve[j] = layer_trace(Wt, x_mc)[-1][:, coord].mean()
    fig = go.Figure()
    for m in range(n_fixed):
        fig.add_trace(go.Scatter(x=ts.numpy(), y=f_curves[:, m].numpy(), mode="lines",
                                 line=dict(width=1), opacity=0.6, showlegend=(m == 0), name="f(x) (kinked)"))
    fig.add_trace(go.Scatter(x=ts.numpy(), y=F_curve.numpy(), mode="lines",
                             line=dict(color="crimson", width=3), name="F = E_x[f] (smooth)"))
    fig.update_layout(title=f"f is C⁰ (kinks), F is C¹ (smooth) — along W + t·ΔW, coord {coord}",
                      xaxis_title="t", yaxis_title=f"output coord {coord}", template="plotly_white", height=420)
    fig.show()

interact(smoothness_scan, seed=IntSlider(0, 0, 30), coord=IntSlider(0, 0, d - 1),
         t_max=FloatSlider(value=3.0, min=0.5, max=8.0, step=0.5));

## Tie-back

Every design choice in `01_the_method` traces to this notebook:
- `F` is a **smooth** function of `W` (§4) → learnable at all.
- `F` is **heavy-tailed** (§2) → we learn in $\log(1+F)$ space.
- **Width drives concentration** (§2) → `R²` *rises* with width; the moment
  description becomes sufficient, which is why the equivariant *moment-propagator*
  thrives at large width.
- **Depth compounds correlations / Lyapunov spread** (§1) → `R²` *softens* with depth.
- **Width keeps deep nets alive** (§3) → the problem is only non-trivial where the
  network isn't dead.

The forward pass is a random dynamical system; our method is a **learned simulator
of its moment dynamics** that respects the weight-space symmetry. That is the whole
story, end to end.